<a href="https://colab.research.google.com/github/mimia3ung-lang/Blockchain/blob/main/Exercise_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import hashlib
import pickle
import secrets
import statistics
import time


# =========================================================
# 1. CREATE RANDOM BLOCK DATA
# =========================================================

def create_block(block_number):
    """
    Create different block data for each mining experiment.
    """

    return {
        "block_number": block_number,
        "timestamp": time.time(),
        "transactions": [
            secrets.token_hex(8),
            secrets.token_hex(8),
            secrets.token_hex(8)
        ],
        "previous_hash": secrets.token_hex(32),
        "merkle_root": secrets.token_hex(32)
    }


# =========================================================
# 2. MINE ONE BLOCK
# =========================================================

def mine_block(block, difficulty):
    """
    Increment the nonce until the SHA-256 hash begins
    with the required number of leading zeros.
    """

    target = "0" * difficulty
    nonce = 0
    hash_attempts = 0

    pickled_block = pickle.dumps(block)

    start_time = time.perf_counter()

    while True:
        work = hashlib.sha256(
            str(nonce).encode() + pickled_block
        ).hexdigest()

        hash_attempts += 1

        if work.startswith(target):
            break

        nonce += 1

    end_time = time.perf_counter()
    mining_time = end_time - start_time

    return {
        "block_number": block["block_number"],
        "difficulty": difficulty,
        "nonce": nonce,
        "valid_hash": work,
        "hash_attempts": hash_attempts,
        "mining_time": mining_time
    }


# =========================================================
# 3. MINE 30 BLOCKS FOR ONE DIFFICULTY
# =========================================================

def run_experiment(difficulty, number_of_blocks=30):
    """
    Mine 30 different blocks at one difficulty level.
    """

    results = []

    print("\n" + "=" * 100)
    print(
        f"MINING {number_of_blocks} BLOCKS "
        f"AT DIFFICULTY {difficulty}"
    )
    print("=" * 100)

    for block_number in range(1, number_of_blocks + 1):

        block = create_block(block_number)

        result = mine_block(
            block=block,
            difficulty=difficulty
        )

        results.append(result)

        print(
            f"Block {block_number:02d} | "
            f"Time: {result['mining_time']:.6f} seconds | "
            f"Attempts: {result['hash_attempts']:,} | "
            f"Nonce: {result['nonce']:,}"
        )

        print(f"Valid Hash: {result['valid_hash']}")

    return results


# =========================================================
# 4. CALCULATE PERFORMANCE METRICS
# =========================================================

def calculate_metrics(difficulty, results):
    """
    Calculate all metrics required by the assignment.
    """

    mining_times = [
        result["mining_time"]
        for result in results
    ]

    attempts = [
        result["hash_attempts"]
        for result in results
    ]

    total_time = sum(mining_times)
    total_attempts = sum(attempts)

    average_mining_time = statistics.mean(mining_times)
    minimum_mining_time = min(mining_times)
    maximum_mining_time = max(mining_times)

    standard_deviation = statistics.stdev(mining_times)

    average_attempts = statistics.mean(attempts)

    mining_throughput = (
        total_attempts / total_time
        if total_time > 0
        else 0
    )

    theoretical_attempts = 16 ** difficulty

    percentage_difference = (
        (
            average_attempts - theoretical_attempts
        )
        / theoretical_attempts
    ) * 100

    return {
        "difficulty": difficulty,
        "average_time": average_mining_time,
        "minimum_time": minimum_mining_time,
        "maximum_time": maximum_mining_time,
        "standard_deviation": standard_deviation,
        "hashes_per_second": mining_throughput,
        "average_attempts": average_attempts,
        "theoretical_attempts": theoretical_attempts,
        "percentage_difference": percentage_difference
    }


# =========================================================
# 5. DISPLAY SUMMARY TABLE
# =========================================================

def display_summary_table(summary_results):

    print("\n" + "=" * 170)
    print("PROOF-OF-WORK PERFORMANCE SUMMARY")
    print("=" * 170)

    print(
        f"{'Difficulty':<12}"
        f"{'Avg Time (s)':<17}"
        f"{'Min Time (s)':<17}"
        f"{'Max Time (s)':<17}"
        f"{'Std Dev (s)':<17}"
        f"{'Hashes/Second':<20}"
        f"{'Measured Attempts':<22}"
        f"{'Theoretical':<18}"
        f"{'% Difference':<15}"
    )

    print("-" * 170)

    for result in summary_results:
        print(
            f"{result['difficulty']:<12}"
            f"{result['average_time']:<17.6f}"
            f"{result['minimum_time']:<17.6f}"
            f"{result['maximum_time']:<17.6f}"
            f"{result['standard_deviation']:<17.6f}"
            f"{result['hashes_per_second']:<20,.2f}"
            f"{result['average_attempts']:<22,.2f}"
            f"{result['theoretical_attempts']:<18,}"
            f"{result['percentage_difference']:<15.2f}"
        )


# =========================================================
# 6. RUN ALL DIFFICULTY LEVELS
# =========================================================

difficulty_levels = [2, 3, 4, 5]
summary_results = []

for difficulty in difficulty_levels:

    mining_results = run_experiment(
        difficulty=difficulty,
        number_of_blocks=30
    )

    metrics = calculate_metrics(
        difficulty=difficulty,
        results=mining_results
    )

    summary_results.append(metrics)


# =========================================================
# 7. DISPLAY FINAL TABLE
# =========================================================

display_summary_table(summary_results)


MINING 30 BLOCKS AT DIFFICULTY 2
Block 01 | Time: 0.000288 seconds | Attempts: 154 | Nonce: 153
Valid Hash: 007f9e122f13e2851a4afb9ce7370b585ef8a6f0d0e65860e52a97b612472092
Block 02 | Time: 0.000340 seconds | Attempts: 199 | Nonce: 198
Valid Hash: 00b6f4ed90e7e44635167350d9c5943716df16c3b14fc93ab3c45f066c351770
Block 03 | Time: 0.000356 seconds | Attempts: 221 | Nonce: 220
Valid Hash: 0033a970b7630719dedaa90ebb187f04413dd97b37ac250f466c3c435ecf610e
Block 04 | Time: 0.000101 seconds | Attempts: 47 | Nonce: 46
Valid Hash: 006a3ce29e77d0ddb42de2a52f08c85504241ba33483d26a630cd39d25f8079d
Block 05 | Time: 0.001728 seconds | Attempts: 330 | Nonce: 329
Valid Hash: 0010e34d075a100712fe9ed6b3a1dde6e750d74942366001fa0d6b210e6e4b47
Block 06 | Time: 0.000018 seconds | Attempts: 10 | Nonce: 9
Valid Hash: 0070cfb87a34637fb7d826a91a58249dc2f0c3578648cfcf3d291d06b715e3db
Block 07 | Time: 0.000329 seconds | Attempts: 190 | Nonce: 189
Valid Hash: 00f0d9fa9c1e8abe60d74cc147e62e1d6d1c9a8f0086f7784f14a9f5